# Qwen3.5-0.8B → diffusion LM (Mercury-style) on Kaggle

Adapts an autoregressive Qwen into a discrete-diffusion language model using
the DiffuLLaMA/Dream/LLaDA recipe: bidirectional attention (causal→full
annealing) + absorbing-state masked-diffusion objective + confidence-based
parallel denoising.

**Requirements:** a GPU accelerator (Settings → Accelerator → GPU T4 ×2 or
P100) and Internet ON. The model-agnostic core (`diffusion.py`) is the same
one unit-tested on CPU in this repo; only the HuggingFace wiring is here.

**Tip:** run the `supra50m` smoke test first (cell below) to validate the
whole pipeline before downloading Qwen.

In [ ]:
!pip install -q "transformers==5.8.*" "peft>=0.18" accelerate datasets safetensors

## Get the code
Either clone the repo, or upload `diffusion.py` + `qwen_adapt.py` as a Kaggle
dataset / via the file editor. Adjust the path below so both modules are
importable.

In [ ]:
import sys, os
# e.g. after: !git clone <repo> /kaggle/working/MyLittlePony
CODE_DIR = "/kaggle/working/MyLittlePony/experiments/diffusion_port"
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
import diffusion as D
import qwen_adapt as Q
print("core fns:", [f for f in dir(D) if not f.startswith('_')])

## (Optional) smoke test on the in-repo 50M Llama
Validates load → [MASK] → bidirectional → LoRA → diffusion step → denoise on a
tiny real model before touching Qwen. Point `--model` at the supra50m dir.

In [ ]:
import sys
sys.argv = ["qwen_adapt.py",
            "--model", "../shakespeare_port/models/supra50m",
            "--steps", "200", "--bs", "4", "--seq_len", "256",
            "--anneal_steps", "100", "--log_every", "50"]
Q.main()

## The real port: Qwen3.5-0.8B
~6–7GB on a single T4 (bf16 weights 1.6GB + LoRA/optim ~1GB + activations w/
grad-checkpointing ~3GB). Bump `--steps` for better samples.

In [ ]:
import sys
sys.argv = ["qwen_adapt.py",
            "--model", "Qwen/Qwen3.5-0.8B",
            "--steps", "2000", "--bs", "4", "--seq_len", "512",
            "--lr", "1e-4", "--lora_r", "16", "--anneal_steps", "400",
            "--grad_accum", "4", "--dataset", "wikitext",
            "--gen_len", "64", "--gen_steps", "64",
            "--save", "/kaggle/working/diffusion_adapter"]
Q.main()

## Generate from the saved adapter
After training you can reload and sample with the same confidence-based
denoiser at any length / step count.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

name = "Qwen/Qwen3.5-0.8B"
tok = AutoTokenizer.from_pretrained(name)
tok.add_special_tokens({"additional_special_tokens": ["<|mask|>"]})
mask_id = tok.convert_tokens_to_ids("<|mask|>")
base = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16,
                                            attn_implementation="eager")
base.resize_token_embeddings(len(tok))
model = PeftModel.from_pretrained(base, "/kaggle/working/diffusion_adapter").cuda().eval()
restore = Q.force_bidirectional(model)

prompt = tok("In the beginning", return_tensors="pt").input_ids.cuda()
fwd = lambda ids: model(input_ids=ids).logits.float()
out = D.diffusion_generate(fwd, length=prompt.shape[1] + 96, mask_id=mask_id,
                           steps=96, prompt_ids=prompt, temperature=0.4, device="cuda")
print(tok.decode(out[0], skip_special_tokens=True))
restore()